# Hackathon Project: Travel Experience Prediction

**Context:** This notebook aims to predict the `Overall_Experience` of customers traveling. The dataset contains travel and survey data. The goal is to build an accurate machine learning model to classify whether a customer had a positive or negative experience based on various features such as age, travel class, and survey responses.

**Explanation:** Importing required libraries for data manipulation, visualization, and modeling.

In [ ]:
import warnings

warnings.filterwarnings("ignore")
from statsmodels.tools.sm_exceptions import ConvergenceWarning

warnings.simplefilter("ignore", ConvergenceWarning)

# Libraries to help with reading and manipulating data

import pandas as pd
import numpy as np

# Library to split data
from sklearn.model_selection import train_test_split

# libaries to help with data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Removes the limit for the number of displayed columns
pd.set_option("display.max_columns", None)
# Sets the limit for the number of displayed rows
pd.set_option("display.max_rows", 200)
# setting the precision of floating numbers to 5 decimal points
pd.set_option("display.float_format", lambda x: "%.5f" % x)

# To build model for prediction
import statsmodels.stats.api as sms
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
from statsmodels.tools.tools import add_constant
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree
from sklearn.ensemble import RandomForestClassifier

# To tune different models
from sklearn.model_selection import GridSearchCV


# To get diferent metric scores
import sklearn.metrics as metrics
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    recall_score,
    precision_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    precision_recall_curve,
    roc_curve,
    make_scorer,
)

##XG

**Explanation:** Installing necessary packages for our environment.

In [ ]:
!pip install xgboost

**Explanation:** Loading the dataset into a pandas DataFrame for analysis.

In [ ]:
travel = pd.read_csv('Traveldata_train_(1)_(1).csv')
survey = pd.read_csv('Surveydata_train_(1)_(1).csv')
df = pd.merge(travel, survey, on='ID')

Traveltest = pd.read_csv('Traveldata_test_(1)_(1).csv')
surveytest = pd.read_csv('Surveydata_test_(1)_(1).csv')
dftest = pd.merge(Traveltest, surveytest, on='ID')

**Explanation:** Handling missing values by imputing unknown categories for categorical columns and median values for numerical ones.

In [ ]:
cat_cols = df.select_dtypes(include='object').columns
num_cols = df.select_dtypes(exclude='object').columns

df[cat_cols] = df[cat_cols].fillna('Unknown')
dftest[cat_cols] = dftest[cat_cols].fillna('Unknown')

for col in num_cols:
    if col != 'Overall_Experience':
        median = df[col].median()
        df[col].fillna(median, inplace=True)
        dftest[col].fillna(median, inplace=True)

df.drop('ID', axis=1, inplace=True)

**Explanation:** One-hot encoding categorical variables to convert them into a numerical format suitable for the model.

In [ ]:
df = pd.get_dummies(df, drop_first=True)
dftest = pd.get_dummies(dftest, drop_first=True)

# ALIGN
df, dftest = df.align(dftest, join='left', axis=1, fill_value=0)

**Explanation:** Separating the dataset into features (X) and the target variable (y).

In [ ]:
X = df.drop('Overall_Experience', axis=1)
y = df['Overall_Experience']

**Explanation:** Splitting the data into training and validation sets to evaluate the model's generalization.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

**Explanation:** Initializing and training the XGBoost classifier model.

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=600,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1,
    reg_lambda=1,
    random_state=42,
    eval_metric='logloss'
)

xgb.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=600, n_jobs=None,
              num_parallel_tree=None, ...)

**Explanation:** Evaluating the model's accuracy on both the training and validation sets to check for overfitting.

In [ ]:
# Train performance
xgb_train_pred = xgb.predict(X_train)
xgb_train_acc = accuracy_score(y_train, xgb_train_pred)

# Validation performance
xgb_val_pred = xgb.predict(X_val)
xgb_val_acc = accuracy_score(y_val, xgb_val_pred)

print("XGBoost Train Accuracy:", xgb_train_acc)
print("XGBoost Validation Accuracy:", xgb_val_acc)

XGBoost Train Accuracy: 0.9624650676132074
XGBoost Validation Accuracy: 0.9517906336088154


**Explanation:** Iterating over different probability thresholds to find the optimal cutoff that maximizes accuracy.

In [ ]:
proba = xgb.predict_proba(X_val)[:,1]

for t in [0.45, 0.47, 0.5, 0.52, 0.55]:
    pred = (proba > t).astype(int)
    print(f"Threshold {t}: {accuracy_score(y_val, pred)}")

Threshold 0.45: 0.9521614748887476
Threshold 0.47: 0.9521614748887476
Threshold 0.5: 0.9517906336088154
Threshold 0.52: 0.9516846789574063
Threshold 0.55: 0.9515257469802925


**Explanation:** Setting the best probability threshold found during our search.

In [ ]:
BEST_THRESHOLD = 0.47

**Explanation:** Applying the optimal threshold to generate final class predictions.

In [ ]:
final_val_pred = (proba > BEST_THRESHOLD).astype(int)

print("Final Validation Accuracy:", accuracy_score(y_val, final_val_pred))

Final Validation Accuracy: 0.9521614748887476


**Explanation:** Initializing and training the XGBoost classifier model.

In [ ]:
from xgboost import XGBClassifier

xgb2 = XGBClassifier(
    n_estimators=2000,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1,
    reg_lambda=1,
    random_state=42,
    eval_metric='logloss',

    early_stopping_rounds=50   #ADD
)

xgb2.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=50,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=2000, n_jobs=None,
              num_parallel_tree=None, ...)

**Explanation:** Evaluating the model's accuracy on both the training and validation sets to check for overfitting.

In [ ]:
# Train performance
xgb_train_pred = xgb2.predict(X_train)
xgb_train_acc = accuracy_score(y_train, xgb_train_pred)

# Validation performance
xgb_val_pred = xgb2.predict(X_val)
xgb_val_acc = accuracy_score(y_val, xgb_val_pred)

print("XGBoost Train Accuracy:", xgb_train_acc)
print("XGBoost Validation Accuracy:", xgb_val_acc)

XGBoost Train Accuracy: 0.9794312808762565
XGBoost Validation Accuracy: 0.9541746132655223


**Explanation:** Iterating over different probability thresholds to find the optimal cutoff that maximizes accuracy.

In [ ]:
proba = xgb2.predict_proba(X_val)[:,1]

for t in [0.45, 0.47, 0.5, 0.52, 0.55]:
    pred = (proba > t).astype(int)
    print(f"Threshold {t}: {accuracy_score(y_val, pred)}")

Threshold 0.45: 0.953962703962704
Threshold 0.47: 0.9539097266369994
Threshold 0.5: 0.9541746132655223
Threshold 0.52: 0.9547573638482729
Threshold 0.55: 0.9547043865225684


**Explanation:** Setting the best probability threshold found during our search.

In [ ]:
BEST_THRESHOLD = 0.52

**Explanation:** Applying the optimal threshold to generate final class predictions.

In [ ]:
final_val_pred = (proba > BEST_THRESHOLD).astype(int)

print("Final Validation Accuracy:", accuracy_score(y_val, final_val_pred))

Final Validation Accuracy: 0.9547573638482729


**Explanation:** Extracting and sorting feature importances to identify which variables have the most predictive power.

In [ ]:
importances = xgb2.feature_importances_

feat_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

**Explanation:** Exploring the dataset to understand its structure, shape, and basic statistical properties.

In [ ]:
top_features_50 = feat_df.head(50)['Feature']
top_features_70 = feat_df.head(70)['Feature']
top_features_100 = feat_df.head(100)['Feature']

**Explanation:** Selecting the top most important features to perform feature selection.

In [ ]:
X_train_50 = X_train[top_features_50]
X_val_50 = X_val[top_features_50]

X_train_70 = X_train[top_features_70]
X_val_70 = X_val[top_features_70]

**Explanation:** Training an XGBoost classifier using only the selected top features to improve generalization and speed.

In [ ]:
xgb_fs = XGBClassifier(
    n_estimators=2000,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss',
    early_stopping_rounds=50
)

xgb_fs.fit(
    X_train_70, y_train,   #ADD
    eval_set=[(X_val_70, y_val)],
    verbose=False
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=50,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=2000, n_jobs=None,
              num_parallel_tree=None, ...)

**Explanation:** Evaluating the model's accuracy on both the training and validation sets to check for overfitting.

In [ ]:
# Train performance
xgb_fs_train_pred = xgb_fs.predict(X_train_70)
xgb_fs_train_acc = accuracy_score(y_train, xgb_fs_train_pred)

# Validation performance
xgb_fs_val_pred = xgb_fs.predict(X_val_70)
xgb_fs_val_acc = accuracy_score(y_val, xgb_fs_val_pred)

print("XGB FS Train Accuracy:", xgb_fs_train_acc)
print("XGB FS Validation Accuracy:", xgb_fs_val_acc)

XGB FS Train Accuracy: 0.9768883355628253
XGB FS Validation Accuracy: 0.9549162958253867


**Explanation:** Iterating over different probability thresholds to find the optimal cutoff that maximizes accuracy.

In [ ]:
proba_fs = xgb_fs.predict_proba(X_val_70)[:,1]

for t in [0.43, 0.44, 0.45, 0.46, 0.47, 0.48, 0.49, 0.50, 0.51, 0.52, 0.53]:
    pred = (proba_fs > t).astype(int)
    print(f"{t}: {accuracy_score(y_val, pred)}")

0.43: 0.9532210214028396
0.44: 0.9535918626827717
0.45: 0.954227590591227
0.46: 0.9545454545454546
0.47: 0.9549162958253867
0.48: 0.9547573638482729
0.49: 0.9545984318711591
0.5: 0.9549162958253867
0.51: 0.9550222504767959
0.52: 0.9552341597796143
0.53: 0.9550752278025005


**Explanation:** Setting the best probability threshold found during our search.

In [ ]:
BEST_THRESHOLD = 0.52

**Explanation:** Applying the optimal threshold to generate final class predictions.

In [ ]:
final_val_pred = (proba_fs > BEST_THRESHOLD).astype(int)

print("Final Validation Accuracy:", accuracy_score(y_val, final_val_pred))

Final Validation Accuracy: 0.9552341597796143


**Explanation:** Selecting the top most important features to perform feature selection.

In [ ]:
xgb_fs.fit(
    X[top_features_70], y,
    eval_set=[(X[top_features_70], y)],
    verbose=False
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=50,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=2000, n_jobs=None,
              num_parallel_tree=None, ...)

**Explanation:** Selecting the top most important features to perform feature selection.

In [ ]:
# salvar ordem usada no treino
colunas_do_treino = X[top_features_70].columns.tolist()

**Explanation:** Reindexing the test dataset to perfectly match the feature columns of the training set.

In [ ]:
# alinhar colunas
X_test_final = dftest.copy()

# garantir mesmas colunas e ordem
X_test_final = X_test_final.reindex(columns=colunas_do_treino, fill_value=0)

**Explanation:** Reindexing the test dataset to perfectly match the feature columns of the training set.

In [ ]:
# alinhar colunas
X_test_final = dftest.copy()

# garantir mesmas colunas e ordem
X_test_final = X_test_final.reindex(columns=colunas_do_treino, fill_value=0)

**Explanation:** Casting the test features to float to ensure compatibility with the trained model.

In [ ]:
X_test_final = X_test_final.astype(float)

**Explanation:** Generating probability predictions for the test set.

In [ ]:
proba_test = xgb_fs.predict_proba(X_test_final)[:,1]

**Explanation:** Setting the best probability threshold found during our search.

In [ ]:
BEST_THRESHOLD = 0.47

final_pred = (proba_test > BEST_THRESHOLD).astype(int)

**Explanation:** Loading the dataset into a pandas DataFrame for analysis.

In [ ]:
sample_sub = pd.read_csv('Sample_Submission_(1)_(1).csv')

sample_sub['Overall_Experience'] = final_pred

sample_sub.to_csv('final_submission_xgboost.csv', index=False)

print("✅ XGBoost submission generated successfully!")

✅ XGBoost submission generated successfully!


**Explanation:** Loading the dataset into a pandas DataFrame for analysis.

In [ ]:
teste = pd.read_csv('final_submission_xgboost.csv')

**Explanation:** Previewing the generated data structure.

In [ ]:
teste.head()

,ID,Overall_Experience
0,99900001,1
1,99900002,1
2,99900003,1
3,99900004,0
4,99900005,1


##CAT

**Explanation:** Installing necessary packages for our environment.

In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.8 MB/s eta 0:00:00


**Explanation:** Loading the dataset into a pandas DataFrame for analysis.

In [ ]:
travel = pd.read_csv('Traveldata_train_(1)_(1).csv')
survey = pd.read_csv('Surveydata_train_(1)_(1).csv')
df = pd.merge(travel, survey, on='ID')

Traveltest = pd.read_csv('Traveldata_test_(1)_(1).csv')
surveytest = pd.read_csv('Surveydata_test_(1)_(1).csv')
dftest = pd.merge(Traveltest, surveytest, on='ID')

**Explanation:** Exploring the dataset to understand its structure, shape, and basic statistical properties.

In [ ]:
df.dtypes

,0
ID,int64
Gender,object
Customer_Type,object
Age,float64
Type_Travel,object
Travel_Class,object
Travel_Distance,int64
Departure_Delay_in_Mins,float64
Arrival_Delay_in_Mins,float64
Overall_Experience,int64


**Explanation:** Separating the dataset into features (X) and the target variable (y).

In [ ]:
X = df.drop('Overall_Experience', axis=1)
y = df['Overall_Experience']

**Explanation:** Defining the list of categorical features explicitly for the CatBoost model.

In [ ]:
cat_features = [
    'Gender', 'Customer_Type', 'Type_Travel', 'Travel_Class', 'Seat_Class',
    'Seat_Comfort', 'Arrival_Time_Convenient', 'Catering', 'Platform_Location',
    'Onboard_Wifi_Service', 'Onboard_Entertainment', 'Online_Support',
    'Ease_of_Online_Booking', 'Onboard_Service', 'Legroom', 'Baggage_Handling',
    'CheckIn_Service', 'Cleanliness', 'Online_Boarding'
]

# Garantindo que o Python entenda que são categorias (strings)
for col in cat_features:
    X[col] = X[col].astype(str)

**Explanation:** Splitting the data into training and validation sets to evaluate the model's generalization.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y  # Mantém a mesma proporção de Satisfeitos/Insatisfeitos
)

**Explanation:** Initializing and training the CatBoost classifier, utilizing early stopping to prevent overfitting.

In [ ]:
from catboost import CatBoostClassifier

model_cat = CatBoostClassifier(
    iterations=2000,           # Mais árvores permitem aprender detalhes finos
    learning_rate=0.03,        # Passo lento para maior precisão
    depth=6,                   # Profundidade padrão (ideal para evitar overfitting)
    l2_leaf_reg=5,             # Regularização para manter o modelo simples
    eval_metric='Accuracy',    # Métrica da competição
    random_seed=42,
    verbose=100                # Reporta o progresso a cada 100 árvores
)

# Treinamento com "Parada Antecipada"
model_cat.fit(
    X_train, y_train,
    cat_features=cat_features,
    eval_set=(X_val, y_val),
    early_stopping_rounds=100  # Para se a acurácia de validação parar de subir
)

0:	learn: 0.8396620	test: 0.8413859	best: 0.8413859 (0)	total: 438ms	remaining: 14m 34s
100:	learn: 0.9345986	test: 0.9299110	best: 0.9299110 (100)	total: 27.4s	remaining: 8m 35s
200:	learn: 0.9459624	test: 0.9419369	best: 0.9419369 (198)	total: 58s	remaining: 8m 38s
300:	learn: 0.9510483	test: 0.9465459	best: 0.9465459 (299)	total: 1m 26s	remaining: 8m 6s
400:	learn: 0.9534191	test: 0.9486120	best: 0.9486120 (356)	total: 1m 53s	remaining: 7m 32s
500:	learn: 0.9551144	test: 0.9500954	best: 0.9500954 (483)	total: 2m 21s	remaining: 7m 3s
600:	learn: 0.9574454	test: 0.9520555	best: 0.9523204 (593)	total: 2m 50s	remaining: 6m 36s
700:	learn: 0.9586109	test: 0.9529032	best: 0.9529032 (692)	total: 3m 20s	remaining: 6m 11s
800:	learn: 0.9596440	test: 0.9536448	best: 0.9536978 (795)	total: 3m 49s	remaining: 5m 42s
900:	learn: 0.9604784	test: 0.9540157	best: 0.9541216 (895)	total: 4m 17s	remaining: 5m 14s
1000:	learn: 0.9611406	test: 0.9544925	best: 0.9544925 (996)	total: 4m 46s	remaining: 4m 4

CatBoostClassifier(depth=6, eval_metric='Accuracy', iterations=2000, l2_leaf_reg=5, learning_rate=0.03, random_seed=42, verbose=100)

**Explanation:** Evaluating the model's accuracy on both the training and validation sets to check for overfitting.

In [ ]:
# Train performance
cat_train_pred = model_cat.predict(X_train)
cat_train_acc = accuracy_score(y_train, cat_train_pred)

# Validation performance
cat_val_pred = model_cat.predict(X_val)
cat_val_acc = accuracy_score(y_val, cat_val_pred)

print(f"CatBoost Train Accuracy: {cat_train_acc:.5f}")
print(f"CatBoost Validation Accuracy: {cat_val_acc:.5f}")

CatBoost Train Accuracy: 0.96485
CatBoost Validation Accuracy: 0.95719


**Explanation:** Iterating over different probability thresholds to find the optimal cutoff that maximizes accuracy.

In [ ]:
proba_cat = model_cat.predict_proba(X_val)[:, 1]

# Dicionário para guardar os resultados e encontrar o máximo automaticamente
results = {}

print("Busca de Threshold:")
for t in [0.43, 0.44, 0.45, 0.46, 0.47, 0.48, 0.49, 0.50, 0.51, 0.52, 0.53, 0.54, 0.55]:
    pred = (proba_cat > t).astype(int)
    acc = accuracy_score(y_val, pred)
    results[t] = acc
    print(f"T={t:.2f} | Accuracy: {acc:.5f}")

# Print do melhor absoluto
best_t = max(results, key=results.get)
print(f"\nBest Threshold: {best_t} Accuracy: {results[best_t]:.5f}")

Busca de Threshold:
T=0.43 | Accuracy: 0.95545
T=0.44 | Accuracy: 0.95539
T=0.45 | Accuracy: 0.95592
T=0.46 | Accuracy: 0.95587
T=0.47 | Accuracy: 0.95608
T=0.48 | Accuracy: 0.95640
T=0.49 | Accuracy: 0.95651
T=0.50 | Accuracy: 0.95719
T=0.51 | Accuracy: 0.95704
T=0.52 | Accuracy: 0.95682
T=0.53 | Accuracy: 0.95698
T=0.54 | Accuracy: 0.95677
T=0.55 | Accuracy: 0.95693

Best Threshold: 0.5 Accuracy: 0.95719


**Explanation:** Loading the dataset into a pandas DataFrame for analysis.

In [ ]:
# 1. Pegar a lista exata de colunas que o modelo espera (na ordem correta)
colunas_treino = X.columns.tolist()

# 2. Reorganizar o X_test_final para seguir EXATAMENTE essa lista
# Isso remove colunas extras (como 'ID') e coloca as outras no lugar certo
X_test_final = dftest[colunas_treino].copy()

# 3. Garantir que as categorias sejam strings (novamente, por segurança)
for col in cat_features:
    X_test_final[col] = X_test_final[col].astype(str)

# 4. Agora o predict deve funcionar sem erro de índice
final_preds = model_cat.predict(X_test_final)

# 5. Colocar na planilha de submissão
sample_sub = pd.read_csv('Sample_Submission_(1)_(1).csv')
sample_sub['Overall_Experience'] = final_preds
sample_sub.to_csv('final_submission_catboost.csv', index=False)

print("✅ Sucesso! O erro de índice foi resolvido alinhando as colunas.")

✅ Sucesso! O erro de índice foi resolvido alinhando as colunas.


**Explanation:** Loading the dataset into a pandas DataFrame for analysis.

In [ ]:
# 1. Recuperar a ordem exata de colunas do treino
# (Assumindo que 'X' é o dataframe que você usou no model.fit)
colunas_do_treino = X.columns.tolist()

# 2. Reordenar e Filtrar o teste
# Isso remove o 'ID' automaticamente se ele não estiver em colunas_do_treino
X_test_final = dftest[colunas_do_treino].copy()

# 3. Garantir o tipo String para as categóricas (Essencial para o CatBoost)
for col in cat_features:
    X_test_final[col] = X_test_final[col].astype(str)

# 4. Gerar as predições (Agora sem erro de índice)
final_preds = model_cat.predict(X_test_final)

# 5. Criar a submissão usando a planilha oficial como molde
sample_sub = pd.read_csv('Sample_Submission_(1)_(1).csv')
sample_sub['Overall_Experience'] = final_preds

# 6. Exportar
sample_sub.to_csv('final_submission_catboost.csv', index=False)

print("✅ Planilha gerada com sucesso alinhando as colunas!")

✅ Planilha gerada com sucesso alinhando as colunas!


**Explanation:** Loading the dataset into a pandas DataFrame for analysis.

In [ ]:
testhelp = pd.read_csv('final_submission_catboost.csv')

**Explanation:** Previewing the generated data structure.

In [ ]:
testhelp.head()

,ID,Overall_Experience
0,99900001,1
1,99900002,1
2,99900003,1
3,99900004,0
4,99900005,1


**Explanation:** Initializing and training the CatBoost classifier, utilizing early stopping to prevent overfitting.

In [ ]:
from catboost import CatBoostClassifier

model_cat = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=5,
    eval_metric='Accuracy',
    random_seed=42,
    verbose=100
)

model_cat.fit(
    X, y,   # 🔥 TODOS os dados
    cat_features=cat_features,
    eval_set=(X, y),  # opcional aqui
    early_stopping_rounds=100
)

0:	learn: 0.8456966	test: 0.8461734	best: 0.8461734 (0)	total: 649ms	remaining: 21m 37s
100:	learn: 0.9341061	test: 0.9344028	best: 0.9344028 (100)	total: 49.8s	remaining: 15m 36s
200:	learn: 0.9452950	test: 0.9456447	best: 0.9456871 (199)	total: 1m 31s	remaining: 13m 41s
300:	learn: 0.9505187	test: 0.9510166	best: 0.9510166 (300)	total: 2m 15s	remaining: 12m 42s
400:	learn: 0.9536549	test: 0.9540788	best: 0.9541106 (397)	total: 2m 57s	remaining: 11m 47s
500:	learn: 0.9564734	test: 0.9567806	best: 0.9567806 (500)	total: 3m 40s	remaining: 10m 58s
600:	learn: 0.9580627	test: 0.9582746	best: 0.9582746 (599)	total: 4m 23s	remaining: 10m 13s
700:	learn: 0.9591964	test: 0.9592494	best: 0.9592600 (698)	total: 5m 6s	remaining: 9m 27s
800:	learn: 0.9598534	test: 0.9601288	best: 0.9601500 (792)	total: 5m 48s	remaining: 8m 41s
900:	learn: 0.9606586	test: 0.9609129	best: 0.9609129 (900)	total: 6m 31s	remaining: 7m 57s
1000:	learn: 0.9614003	test: 0.9615698	best: 0.9615698 (1000)	total: 7m 15s	rema

CatBoostClassifier(depth=6, eval_metric='Accuracy', iterations=2000, l2_leaf_reg=5, learning_rate=0.03, random_seed=42, verbose=100)

**Explanation:** Saving the exact column order used during training to ensure the test set is formatted identically.

In [ ]:
colunas_do_treino = X.columns.tolist()

**Explanation:** Filtering the test dataset to include only the columns that were used during training.

In [ ]:
X_test_final = dftest[colunas_do_treino].copy()

**Explanation:** Ensuring categorical columns are cast as strings, which is required by certain models like CatBoost.

In [ ]:
for col in cat_features:
    X_test_final[col] = X_test_final[col].astype(str)

**Explanation:** Setting the best probability threshold found during our search.

In [ ]:
BEST_THRESHOLD = 0.50

final_preds = (proba_test > BEST_THRESHOLD).astype(int)

**Explanation:** Loading the dataset into a pandas DataFrame for analysis.

In [ ]:
sample_sub = pd.read_csv('Sample_Submission_(1)_(1).csv')

sample_sub['Overall_Experience'] = final_preds

sample_sub.to_csv('final_submission_catboost2.csv', index=False)

print("✅ Final CatBoost submission generated successfully!")

✅ Final CatBoost submission generated successfully!


**Explanation:** Triggering the download of the generated submission file.

In [ ]:
from google.colab import files
files.download('final_submission_catboost2.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Explanation:** Saving the exact column order used during training to ensure the test set is formatted identically.

In [ ]:
colunas_xgb = X_train_70.columns.tolist()

**Explanation:** Reindexing the test dataset to perfectly match the feature columns of the training set.

In [ ]:
X_test_final_xgb = dftest.reindex(columns=colunas_xgb, fill_value=0)

**Explanation:** Extracting probabilities from both CatBoost and XGBoost models to prepare for ensembling.

In [ ]:
proba_cat = model_cat.predict_proba(X_test_final)[:,1]
proba_xgb = xgb_fs.predict_proba(X_test_final_xgb)[:,1]

**Explanation:** Casting the test features to float to ensure compatibility with the trained model.

In [ ]:
X_test_final_xgb = X_test_final_xgb.astype(float)

**Explanation:** Executing data processing and modeling steps.

In [ ]:
proba_xgb = xgb_fs.predict_proba(X_test_final_xgb)[:,1]

**Explanation:** Asserting that the columns in the test set perfectly match the columns in the training set.

In [ ]:
assert list(X_test_final_xgb.columns) == list(X_train_70.columns)

**Explanation:** Applying the optimal threshold to generate final class predictions.

In [ ]:
proba_cat = model_cat.predict_proba(X_test_final)[:,1]
proba_xgb = xgb_fs.predict_proba(X_test_final_xgb)[:,1]

proba_ensemble = 0.7 * proba_cat + 0.3 * proba_xgb

final_preds = (proba_ensemble > 0.50).astype(int)

**Explanation:** Applying the optimal threshold to generate final class predictions.

In [ ]:
proba_cat = model_cat.predict_proba(X_test_final)[:,1]
proba_xgb = xgb_fs.predict_proba(X_test_final_xgb)[:,1]

proba_ensemble = 0.7 * proba_cat + 0.3 * proba_xgb

final_preds = (proba_ensemble > 0.50).astype(int)

**Explanation:** Exporting the final predictions to a CSV file formatted for the competition submission.

In [ ]:
sample_sub['Overall_Experience'] = final_preds
sample_sub.to_csv('final_submission_ensemble.csv', index=False)

**Explanation:** Triggering the download of the generated submission file.

In [ ]:
files.download('final_submission_ensemble.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>